# EXP-013 — Gộp head (soup) + cân bằng dự đoán trên tập test
Chỉ test, không train, GPU nào cũng được. Dùng checkpoint có sẵn: EXP-000, EXP-004 anchor e30, EXP-010 plain e5/e10 (3 seed).

- **(A) Soup:** trung bình ma trận head W của nhiều seed/epoch → 1 head. So với **ensemble đầu ra** (trung bình similarity của các head).
- **(B) Cân bằng (transductive):** `center` = trừ trung bình điểm theo lớp trên tập test; `sinkhorn` = ép tỉ lệ dự đoán các lớp gần đều (τ = 0.01, 20 vòng). Dùng toàn bộ tập test lúc dự đoán → báo cáo **riêng**, không trộn với zero-shot chuẩn.

**Tiêu chí (chọn trước):**
- (A) soup3 − mean 3 seed ≥ +0.3 Mean4 UAR ở ≥ 2/3 họ head (EXP-000, EXP-004, EXP-010 e5), không dataset nào < −0.5.
- (B) sinkhorn − none ≥ +1.0 Mean4 UAR trên EXP-010 e5 **và** EXP-000 (mean seed, P0).

Ensemble Five, P0 là cấu hình chính.

# Cấu hình

In [ ]:
SEEDS = [1, 2, 3]
BRANCH = "exp/012-qwen-teacher"   # chỉ cần models/clip, models/Text.py, data_loader (có sẵn từ EXP-009)
TEST_ZIP = "/content/drive/MyDrive/Datasets/static_test_sets.zip"
ROOT = "/content/drive/MyDrive/ExpCLIP_output"
OUT_DIR = f"{ROOT}/exp013"
EXP004_DIR = f"{ROOT}/exp004"
EXP010_DIR = f"{ROOT}/exp010"
TAU, SINK_ITERS = 0.01, 20

import os, glob

# Mount Drive, clone repo, giải nén test

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(OUT_DIR, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv
if not os.path.isdir("/content/Exp-CLIP"):
    !git clone -b $BRANCH https://github.com/huongng389/Exp-CLIP.git /content/Exp-CLIP
%cd /content/Exp-CLIP
!git checkout $BRANCH && git pull
!git log --oneline -1   # ghi lại commit vào research_log
!mkdir -p /content/static_test_sets
!unzip -q -n "$TEST_ZIP" -d /content/static_test_sets
!pip install -q ftfy "setuptools<81"

Mounted at /content/drive
name, memory.total [MiB]
Tesla T4, 15360 MiB
Cloning into '/content/Exp-CLIP'...
remote: Enumerating objects: 486, done.
remote: Counting objects: 100% (486/486), done.
remote: Compressing objects: 100% (205/205), done.
remote: Total 486 (delta 352), reused 410 (delta 277), pack-reused 0 (from 0)
Receiving objects: 100% (486/486), 32.38 MiB | 22.63 MiB/s, done.
Resolving deltas: 100% (352/352), done.
/content/Exp-CLIP
Already on 'exp/012-qwen-teacher'
Your branch is up to date with 'origin/exp/012-qwen-teacher'.
Already up to date.
558dba2 (HEAD -> exp/012-qwen-teacher, origin/exp/012-qwen-teacher) EXP-012: Qwen2.5-VL teacher targets + head output dim from target


# Checkpoint

In [ ]:
def find_ckpt(root, name):
    hits = glob.glob(f"{root}/**/{name}", recursive=True)
    return hits[0] if hits else None

FAMILIES = {   # họ head → {seed: đường dẫn}
    "EXP-000": {s: find_ckpt(ROOT, f"expclip_l14_face_seed{s}-model.pth") for s in SEEDS},
    "EXP-004 e30": {s: f"{EXP004_DIR}/exp004_anchor_crossA_l1_l14_seed{s}-e30-model.pth" for s in SEEDS},
    "EXP-010 e5": {s: f"{EXP010_DIR}/exp010_affnet_plain_l14_seed{s}-e5-model.pth" for s in SEEDS},
    "EXP-010 e10": {s: f"{EXP010_DIR}/exp010_affnet_plain_l14_seed{s}-e10-model.pth" for s in SEEDS},
}
for f, d in FAMILIES.items():
    for s, p in list(d.items()):
        if p is None or not os.path.exists(p):
            print(f"CẢNH BÁO: {f} seed {s}: không thấy checkpoint ({p}) → bỏ qua")
            del d[s]
    print(f, "→", len(d), "seed")
PAPER = "/content/Exp-CLIP/checkpoint/ExpCLIP_L14_model.pth"

EXP-000 → 3 seed
EXP-004 e30 → 3 seed
EXP-010 e5 → 3 seed
EXP-010 e10 → 3 seed


# Feature CLIP của ảnh test (P0 × gốc/lật) và prompt

In [ ]:
import sys, re, json, numpy as np, pandas as pd, torch, torch.nn.functional as F
import torchvision.datasets as datasets
sys.path.insert(0, "/content/Exp-CLIP")
from models.clip import clip
from models.Text import FER_prompt_list, FER_prompt_type_list
from data_loader.image_transform import image_transform

DATASETS = ["RAFDB", "AffectNet7", "AffectNet8", "FERPlus"]
N_CLS = {"RAFDB": 7, "AffectNet7": 7, "AffectNet8": 8, "FERPlus": 8}
PROMPT_NAMES = ["Expression Ensemble Five", "Expression Ensemble Ten"]
BASE_PROMPT = "Expression Ensemble Five"
CFG_NAMES = ["P0", "P0+flip"]
BALANCES = ["none", "center", "sinkhorn"]

PROMPTS = {pn: pl for pn, pl in zip(FER_prompt_type_list, FER_prompt_list) if pn in PROMPT_NAMES}
clip_model, _ = clip.load("ViT-L/14", "cuda")
clip_model.eval()
img, y, classes, txt = {}, {}, {}, {}   # img[ds] = (feat gốc, feat lật)
with torch.no_grad():
    for ds_name in DATASETS:
        ds = datasets.ImageFolder(f"/content/static_test_sets/{ds_name}/test", image_transform("raw"))
        loader = torch.utils.data.DataLoader(ds, batch_size=256, shuffle=False, num_workers=4)
        f0, f1 = [], []
        for x, _ in loader:
            x = x.cuda()
            f0.append(clip_model.encode_image(x).float().cpu())
            f1.append(clip_model.encode_image(x.flip(-1)).float().cpu())   # lật ngang
        img[ds_name] = (torch.cat(f0), torch.cat(f1))
        y[ds_name] = np.array(ds.targets)
        classes[ds_name] = [c.split("_", 1)[1] for c in ds.classes]   # 0_Happiness → Happiness
        for pn, pl in PROMPTS.items():
            txt[(ds_name, pn)] = clip_model.encode_text(
                clip.tokenize(pl[ds_name], context_length=77, truncate=True).cuda()).float().cpu()
        print(ds_name, tuple(img[ds_name][0].shape), classes[ds_name], np.bincount(y[ds_name]))

/content/Exp-CLIP/models/clip/clip.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import packaging
100%|███████████████████████████████████████| 890M/890M [00:32<00:00, 29.1MiB/s]
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 

RAFDB (3068, 768) ['Happiness', 'Sadness', 'Neutral', 'Anger', 'Surprise', 'Disgust', 'Fear'] [1185  478  680  162  329  160   74]
AffectNet7 (3500, 768) ['Happiness', 'Sadness', 'Neutral', 'Anger', 'Surprise', 'Disgust', 'Fear'] [500 500 500 500 500 500 500]
AffectNet8 (4000, 768) ['Happiness', 'Sadness', 'Neutral', 'Anger', 'Surprise', 'Disgust', 'Fear', 'Contempt'] [500 500 500 500 500 500 500 500]
FERPlus (3573, 768) ['Happiness', 'Sadness', 'Neutral', 'Anger', 'Surprise', 'Disgust', 'Fear', 'Contempt'] [ 929  449 1274  322  450   21   98   30]


# Điểm lớp, soup, ensemble, cân bằng
- **soup**: W = trung bình các W, rồi tính như 1 head.
- **ens**: trung bình similarity (đã chuẩn hoá) của từng head.
- Điểm lớp = cos trung bình qua các prompt của lớp (như `test.py`); cân bằng áp lên ma trận điểm (N ảnh × C lớp).

In [ ]:
def load_W(path):
    sd = torch.load(path, map_location="cpu")
    return next(v for k, v in sd.items() if "mlp.weight" in k).float()   # (4096, 768)

def class_scores(W, ds_name, cfg, pn):
    f0, f1 = img[ds_name]
    i, t = ((f0 + f1) / 2 if cfg.endswith("+flip") else f0), txt[(ds_name, pn)]   # TTA: trung bình trước head
    if W is not None:
        i, t = i @ W.T, t @ W.T
    i, t = F.normalize(i, dim=-1), F.normalize(t, dim=-1)
    return (i @ t.T).view(len(i), N_CLS[ds_name], -1).mean(-1).double().numpy()   # (N, C)

def head_scores(Ws, mode, ds_name, cfg, pn):
    if Ws is None:
        return class_scores(None, ds_name, cfg, pn)
    if mode == "soup":
        return class_scores(torch.stack(Ws).mean(0), ds_name, cfg, pn)
    return np.mean([class_scores(W, ds_name, cfg, pn) for W in Ws], axis=0)   # ens

def balance(S, how):
    if how == "none":
        return S
    if how == "center":
        return S - S.mean(0, keepdims=True)
    # sinkhorn: phân phối dự đoán mỗi ảnh có tổng 1, mỗi lớp nhận tổng N/C
    Q = np.exp((S - S.max(1, keepdims=True)) / TAU)
    for _ in range(SINK_ITERS):
        Q /= Q.sum(0, keepdims=True)
        Q /= Q.sum(1, keepdims=True)
    return Q

def metrics(y_, p_, n_cls):
    acc = np.array([(p_[y_ == c] == c).mean() * 100 for c in range(n_cls)])
    frac = np.bincount(p_, minlength=n_cls) / len(p_) * 100
    return {"UAR": acc.mean(), "WAR": (p_ == y_).mean() * 100, "acc": acc, "frac": frac}

In [ ]:
Wf = {f: {s: load_W(p) for s, p in d.items()} for f, d in FAMILIES.items()}
MODELS = {"CLIP gốc": [(0, None, "soup")], "Paper": [(0, [load_W(PAPER)], "soup")]}   # tên → [(seed, Ws, mode)]
for f, d in Wf.items():
    MODELS[f] = [(s, [W], "soup") for s, W in d.items()]                  # từng seed
    MODELS[f"{f} soup3"] = [(0, list(d.values()), "soup")]
    MODELS[f"{f} ens3"] = [(0, list(d.values()), "ens")]
MODELS["EXP-010 soup e5+e10"] = [(0, list(Wf["EXP-010 e5"].values()) + list(Wf["EXP-010 e10"].values()), "soup")]
MODELS["soup chéo 004e30+010e5"] = [(0, list(Wf["EXP-004 e30"].values()) + list(Wf["EXP-010 e5"].values()), "soup")]
MODELS["ens chéo 004e30+010e5"] = [(0, list(Wf["EXP-004 e30"].values()) + list(Wf["EXP-010 e5"].values()), "ens")]

rows = []
for m, entries in MODELS.items():
    for s, Ws, mode in entries:
        for ds_name in DATASETS:
            for cfg in CFG_NAMES:
                for pn in PROMPT_NAMES:
                    S = head_scores(Ws, mode, ds_name, cfg, pn)
                    for b in BALANCES:
                        r = metrics(y[ds_name], balance(S, b).argmax(1), N_CLS[ds_name])
                        rows.append({"model": m, "seed": s, "dataset": ds_name, "config": cfg, "prompt": pn, "balance": b,
                                     "UAR": r["UAR"], "WAR": r["WAR"],
                                     **{f"acc_{c}": a for c, a in zip(classes[ds_name], r["acc"])},
                                     **{f"frac_{c}": a for c, a in zip(classes[ds_name], r["frac"])}})
res = pd.DataFrame(rows)
res.to_csv(f"{OUT_DIR}/exp013_soup_balance-results.csv", index=False)
print(len(res), "dòng →", OUT_DIR)

1200 dòng → /content/drive/MyDrive/ExpCLIP_output/exp013


# Kiểm tra pipeline
Từng seed, P0, `none` phải khớp số `test.py` đã ghi (EXP-000, EXP-004). Lệch ≤ ~0.4 là do fp16.

In [ ]:
REF = {"EXP-000|1|RAFDB": [58.88, 66.88], "EXP-000|1|AffectNet7": [44.09, 44.09], "EXP-000|1|AffectNet8": [39.23, 39.23], "EXP-000|1|FERPlus": [40.65, 51.47], "EXP-000|2|RAFDB": [57.82, 65.78], "EXP-000|2|AffectNet7": [43.69, 43.69], "EXP-000|2|AffectNet8": [38.17, 38.17], "EXP-000|2|FERPlus": [41.08, 53.6], "EXP-000|3|RAFDB": [57.5, 63.3], "EXP-000|3|AffectNet7": [44.74, 44.74], "EXP-000|3|AffectNet8": [39.12, 39.12], "EXP-000|3|FERPlus": [42.93, 55.16], "EXP-004 e30|1|RAFDB": [58.73, 65.94], "EXP-004 e30|1|AffectNet7": [45.83, 45.83], "EXP-004 e30|1|AffectNet8": [39.6, 39.6], "EXP-004 e30|1|FERPlus": [46.14, 52.03], "EXP-004 e30|2|RAFDB": [57.07, 64.77], "EXP-004 e30|2|AffectNet7": [43.97, 43.97], "EXP-004 e30|2|AffectNet8": [38.75, 38.75], "EXP-004 e30|2|FERPlus": [45.63, 52.45], "EXP-004 e30|3|RAFDB": [58.26, 65.06], "EXP-004 e30|3|AffectNet7": [45.46, 45.46], "EXP-004 e30|3|AffectNet8": [39.07, 39.08], "EXP-004 e30|3|FERPlus": [45.34, 51.44]}
chk = []
for k, (u_, w_) in REF.items():
    m, s, ds_name = k.split("|")
    r = res[(res.model == m) & (res.seed == int(s)) & (res.dataset == ds_name) & (res.config == "P0")
            & (res.prompt == BASE_PROMPT) & (res.balance == "none")]
    if len(r):
        chk.append({"key": k, "dUAR": r.UAR.item() - u_, "dWAR": r.WAR.item() - w_})
chk = pd.DataFrame(chk)
print(f"{len(chk)}/{len(REF)} cấu hình so được | max |ΔUAR| = {chk.dUAR.abs().max():.2f}, max |ΔWAR| = {chk.dWAR.abs().max():.2f}")
print(chk.round(2).sort_values("dUAR", key=abs, ascending=False).head(6).to_string(index=False))

# Bảng 1 — (A) Soup so với từng seed (UAR, `none`)

In [ ]:
pd.set_option("display.width", 250)
def fmt(x):
    return f"{x.mean():.2f}" if len(x) == 1 else f"{x.mean():.2f} ± {x.std(ddof=1):.2f}"

def pv(metric="UAR"):
    t = res.pivot_table(index=["model", "seed", "prompt", "config", "balance"], columns="dataset", values=metric)[DATASETS]
    t["Mean4"] = t[DATASETS].mean(1)
    return t
U, Wt = pv("UAR"), pv("WAR")

def get(m, cfg="P0", b="none", pn=BASE_PROMPT, t=None):
    t = U if t is None else t
    return t.loc[m].xs(pn, level="prompt").xs(cfg, level="config").xs(b, level="balance")   # index: seed

COLS = DATASETS + ["Mean4"]
for cfg in CFG_NAMES:
    tab, verdict = {}, []
    for f in FAMILIES:
        single = get(f, cfg)
        tab[f"{f} (từng seed)"] = single.agg(fmt)
        tab[f"{f} max seed"] = single.max().round(2).astype(str)
        for kind in ["soup3", "ens3"]:
            v = get(f"{f} {kind}", cfg).iloc[0]
            tab[f"{f} {kind}"] = v.round(2).astype(str)
            tab[f"  Δ {kind} − mean seed"] = (v - single.mean()).map("{:+.2f}".format)
        if cfg == "P0" and f != "EXP-010 e10":
            d = get(f"{f} soup3", cfg).iloc[0] - single.mean()
            verdict.append((f, d["Mean4"] >= 0.3 and (d[DATASETS] >= -0.5).all(), d["Mean4"], d[DATASETS].min()))
    for m in ["EXP-010 soup e5+e10", "soup chéo 004e30+010e5", "ens chéo 004e30+010e5"]:
        tab[m] = get(m, cfg).iloc[0].round(2).astype(str)
    print(f"--- {cfg}, {BASE_PROMPT}\n{pd.DataFrame(tab).T[COLS].to_string()}\n")
    for f, ok, dm, dmin in verdict:
        print(f"(A) {f}: Δ Mean4 {dm:+.2f}, min Δ dataset {dmin:+.2f} → {'đạt' if ok else 'không đạt'}")
    if verdict:
        n_ok = sum(v[1] for v in verdict)
        print(f"(A) TIÊU CHÍ: {n_ok}/3 họ đạt → {'ĐẠT' if n_ok >= 2 else 'KHÔNG đạt'}\n")

# Bảng 2 — (B) Cân bằng: UAR và WAR (Δ theo cặp seed so với `none`)

In [ ]:
HEADS_B = ["CLIP gốc", "Paper", "EXP-000", "EXP-004 e30", "EXP-010 e5", "EXP-010 e10", "EXP-010 e5 soup3", "soup chéo 004e30+010e5"]
for cfg in CFG_NAMES:
    tab = {}
    for m in HEADS_B:
        for b in BALANCES:
            tab[f"{m} | {b} UAR"] = get(m, cfg, b).agg(fmt)
            if b != "none":
                tab[f"{m} | Δ{b} UAR"] = (get(m, cfg, b) - get(m, cfg)).agg(fmt)
                tab[f"{m} | Δ{b} WAR"] = (get(m, cfg, b, t=Wt) - get(m, cfg, t=Wt)).agg(fmt)
    print(f"--- {cfg}, {BASE_PROMPT}\n{pd.DataFrame(tab).T[COLS].to_string()}\n")

for m in ["EXP-010 e5", "EXP-000"]:
    d = (get(m, "P0", "sinkhorn") - get(m, "P0")).mean()
    print(f"(B) {m}: Δ Mean4 UAR sinkhorn − none = {d['Mean4']:+.2f} → {'đạt' if d['Mean4'] >= 1.0 else 'không đạt'}")
ok_b = all((get(m, "P0", "sinkhorn") - get(m, "P0")).mean()["Mean4"] >= 1.0 for m in ["EXP-010 e5", "EXP-000"])
print(f"(B) TIÊU CHÍ: {'ĐẠT' if ok_b else 'KHÔNG đạt'}")

--- P0, Expression Ensemble Five
dataset                                         RAFDB    AffectNet7    AffectNet8       FERPlus         Mean4
CLIP gốc | none UAR                             52.10         40.69         36.38         38.99         42.04
CLIP gốc | center UAR                           62.11         46.89         40.62         52.10         50.43
CLIP gốc | Δcenter UAR                          10.01          6.20          4.25         13.11          8.39
CLIP gốc | Δcenter WAR                          22.62          6.20          4.25         18.50         12.89
CLIP gốc | sinkhorn UAR                         59.20         47.20         40.73         51.60         49.68
CLIP gốc | Δsinkhorn UAR                         7.09          6.51          4.35         12.61          7.64
CLIP gốc | Δsinkhorn WAR                        14.02          6.51          4.35         15.67         10.14
Paper | none UAR                                57.95         44.23         38.98      

# Bảng 3 — Accuracy và tỉ lệ dự đoán từng lớp (EXP-010 e5, P0, mean seed)

In [ ]:
for ds_name in DATASETS:
    t = {}
    for b in BALANCES:
        r = res[(res.model == "EXP-010 e5") & (res.dataset == ds_name) & (res.prompt == BASE_PROMPT)
                & (res.config == "P0") & (res.balance == b)]
        t[f"acc {b}"] = r[[f"acc_{c}" for c in classes[ds_name]]].mean().values
        t[f"% dự đoán {b}"] = r[[f"frac_{c}" for c in classes[ds_name]]].mean().values
    t["% thật"] = np.bincount(y[ds_name]) / len(y[ds_name]) * 100
    print(f"--- {ds_name}\n{pd.DataFrame(t, index=classes[ds_name]).T.round(1).to_string()}\n")

--- RAFDB
                    Happiness  Sadness  Neutral  Anger  Surprise  Disgust  Fear
acc none                 81.8     55.6     29.7   82.5      63.2     74.4  31.5
% dự đoán none           34.0     17.6     10.7   12.9       9.3     12.6   2.9
acc center               80.7     58.4     31.2   80.0      58.5     75.2  45.9
% dự đoán center         33.6     18.5     11.3   10.4       8.5     12.1   5.6
acc sinkhorn             43.7     49.0     30.1   81.3      57.4     74.0  52.7
% dự đoán sinkhorn       17.1     14.7     16.0   12.4      13.0     13.4  13.4
% thật                   38.6     15.6     22.2    5.3      10.7      5.2   2.4

--- AffectNet7
                    Happiness  Sadness  Neutral  Anger  Surprise  Disgust  Fear
acc none                 74.3     52.5     21.1   37.9      32.1     63.3  28.2
% dự đoán none           15.5     18.6      7.4   12.4      10.7     26.5   8.8
acc center               88.3     58.6     31.8   35.7      26.6     50.0  27.3
% dự đoán cent

# Bảng 4 — Gộp tất cả (Ensemble Five): cấu hình tốt nhất

In [ ]:
best = (U.xs(BASE_PROMPT, level="prompt").reset_index().query("seed == 0")
          .sort_values("Mean4", ascending=False).head(12))
print(best[["model", "config", "balance"] + COLS].round(2).to_string(index=False))

                model  config  balance  RAFDB  AffectNet7  AffectNet8  FERPlus  Mean4
             CLIP gốc P0+flip   center  63.07       46.97       40.48    53.54  51.01
             CLIP gốc      P0   center  62.11       46.89       40.62    52.10  50.43
             CLIP gốc P0+flip sinkhorn  60.20       47.20       40.90    53.01  50.33
      EXP-010 e5 ens3 P0+flip   center  64.07       46.34       39.65    50.29  50.09
ens chéo 004e30+010e5 P0+flip   center  63.98       46.00       39.35    50.64  49.99
             CLIP gốc      P0 sinkhorn  59.20       47.20       40.72    51.60  49.68
     EXP-010 e10 ens3 P0+flip   center  63.48       45.94       39.15    49.71  49.57
         EXP-000 ens3 P0+flip   center  62.99       45.09       38.90    50.84  49.45
      EXP-010 e5 ens3      P0   center  62.88       45.94       39.40    49.22  49.36
ens chéo 004e30+010e5      P0   center  62.55       45.74       39.33    49.64  49.31
     EXP-004 e30 ens3 P0+flip   center  62.57       44

# Lưu JSON (cùng định dạng `research_log/results/`)

In [ ]:
out = {"source": "notebooks/EXP-013_soup_balance.ipynb", "tau": TAU, "sinkhorn_iters": SINK_ITERS,
       "sanity_check": {"max_abs_dUAR": round(float(chk.dUAR.abs().max()), 2), "max_abs_dWAR": round(float(chk.dWAR.abs().max()), 2)},
       "rows": [{"model": r.model, "seed": int(r.seed), "config": r.config, "prompt": r.prompt, "balance": r.balance,
                 "dataset": r.dataset, "UAR": round(r.UAR, 2), "WAR": round(r.WAR, 2)} for r in res.itertuples()]}
with open(f"{OUT_DIR}/EXP-013-soup-balance.json", "w") as f:
    json.dump(out, f, indent=1, ensure_ascii=False)
print(f"{len(out['rows'])} dòng → {OUT_DIR}/EXP-013-soup-balance.json")

1200 dòng → /content/drive/MyDrive/ExpCLIP_output/exp013/EXP-013-soup-balance.json
